# BRFSS 2024 Colorectal Cancer Screening: Data Processing

**Project:** Predicting colorectal cancer screening non-compliance to guide targeted outreach
**Input:** `crc_analytic_dataset.csv` (Data Preparation output -- 227,647 eligible respondents)

This notebook implements proposal step **(c) Data Processing** -- turning the analytic dataset
into model-ready inputs, and validating the assumptions and techniques the modelling notebook
will rely on. It does **not** fit or evaluate a predictive model: no AUROC, no calibration, no
subgroup performance. Those belong in the modelling notebook, which consumes what this notebook
produces.

## What this notebook produces and validates

- The modelling targets (`crc_noncompliant`, plus the two-stage targets prepared for a possible
  follow-up model).
- The M1/M2/M3 predictor domains and the ordinal-vs-nominal encoding plan.
- A single train / calibration / test split, persisted by `respondent_id` so every later notebook
  reuses the same partition.
- A predictor redundancy check (Cramer's V), per proposal step (c).
- The preprocessing pipeline components (logistic-regression encoding, gradient-boosting
  encoding) -- built and smoke-tested here for correctness, not trained for performance.
- Validation that the geographic (leave-states-out) fold structure is sound.
- A KNN-imputed alternative to the primary "Not reported" strategy for `income_group` and
  `bmi_category`, produced and validated mechanically (every cell successfully imputed, sane
  category distributions) -- saved as a lookup table for the modelling notebook's own sensitivity
  comparison, which is where the performance question ("does this change AUROC?") belongs.

## What is deliberately deferred to the modelling notebook

Fitting logistic regression or gradient boosting, the M1/M2/M3 ablation comparison, calibration
and the risk-decile table, subgroup/equity performance, the geographic-CV *performance* numbers,
and the KNN-imputation performance comparison. This notebook prepares and validates everything
those steps need; it doesn't run them.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROCESSED_DIR / "modelling"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

ANALYTIC_PATH = PROCESSED_DIR / "crc_analytic_dataset.csv"
if not ANALYTIC_PATH.exists():
    raise FileNotFoundError(
        f"{ANALYTIC_PATH} not found. Run '1 Data Preparation.ipynb' first."
    )

df = pd.read_csv(
    ANALYTIC_PATH,
    dtype={"respondent_id": "string", "state_fips": "string"},
    low_memory=False,
)
print("Loaded:", ANALYTIC_PATH)
print("Shape:", df.shape)
df.head()

Loaded: ../data/processed/crc_analytic_dataset.csv
Shape: (227647, 27)


,respondent_id,source_row_id,state_fips,validation_group_state,age_exact,_CRCREC3,crc_noncompliant,crc_screening_status,noncompliance_type,age_group,sex,race_ethnicity,education_level,income_group,employment_status,marital_status,urban_rural,insurance_status,personal_doctor,cost_barrier,checkup_recency,general_health,diabetes_status,heart_disease,smoking_status,bmi_category,housing_tenure
0,01-2024000003,3,01,1,59,3,1,Never screened,Never screened,55-59,Male,"White only, non-Hispanic",Attended college or technical school,Not reported,Employed for wages,Member of an unmarried couple,Urban,Insured,No personal doctor,Yes,5 or more years,Very good,No diabetes,CHD or MI not reported,Current smoker - every day,Normal weight,Own
1,01-2024000005,5,01,1,47,3,1,Never screened,Never screened,45-49,Male,"White only, non-Hispanic",Attended college or technical school,"$15,000 to < $25,000",Unable to work,Never married,Urban,Insured,Has personal doctor,No,Within past year,Good,No diabetes,CHD or MI not reported,Never smoked,Normal weight,Own
2,01-2024000006,6,01,1,54,1,0,Up to date,Not applicable - compliant,50-54,Male,"White only, non-Hispanic",Graduated high school,"$100,000 to < $200,000",Employed for wages,Married,Urban,Insured,Has personal doctor,No,Within past year,Good,Diabetes,CHD or MI not reported,Never smoked,Obese,Rent
3,01-2024000007,7,01,1,71,1,0,Up to date,Not applicable - compliant,70-74,Female,"White only, non-Hispanic",Attended college or technical school,"$35,000 to < $50,000",Retired,Married,Rural,Insured,Has personal doctor,No,Within past year,Fair,No diabetes,CHD or MI reported,Never smoked,Obese,Own
4,01-2024000008,8,01,1,68,1,0,Up to date,Not applicable - compliant,65-69,Female,"White only, non-Hispanic",Attended college or technical school,Not reported,Retired,Married,Rural,Insured,Has personal doctor,No,Within past year,Poor,Diabetes,CHD or MI not reported,Never smoked,Obese,Own


## 1. Modelling targets

The primary target already exists from data preparation. We also prepare (but do not model here) the two-stage targets for a possible follow-up notebook, so both are defined from the same source column and can't drift out of sync.

In [2]:
assert df["crc_noncompliant"].isna().sum() == 0
assert set(df["crc_noncompliant"].unique()) == {0, 1}

# Two-stage targets, prepared for a follow-up notebook (not modelled anywhere yet -- built in
# 8 Two-Stage Model.ipynb). Derived via the shared pipeline_components.derive_two_stage_targets
# rather than inline here, since 8 needs the identical derivation and previously duplicated it.
from pipeline_components import derive_two_stage_targets

df["never_screened"], df["overdue_given_screened"], ever_screened_mask = derive_two_stage_targets(df)

print("Primary target rate (crc_noncompliant):", f"{df['crc_noncompliant'].mean():.4f}")
print("Never-screened rate (full cohort):     ", f"{df['never_screened'].mean():.4f}")
print(
    "Overdue rate (among ever-screened only):",
    f"{df.loc[ever_screened_mask, 'overdue_given_screened'].mean():.4f}",
)

assert df["never_screened"].sum() == 43_930
assert ever_screened_mask.sum() == 167_374 + 16_343
assert df.loc[ever_screened_mask, "overdue_given_screened"].sum() == 16_343

Primary target rate (crc_noncompliant): 0.2648
Never-screened rate (full cohort):      0.1930
Overdue rate (among ever-screened only): 0.0890


## 2. Predictor domains and encoding plan

M1/M2/M3 domain structure from notebook 1. Predictors split into **ordinal** (natural order, will be encoded as a rank plus a separate not-reported flag so the missing category doesn't sit inside the ordinal scale) and **nominal** (one-hot for logistic regression; native categorical handling for gradient boosting). `bmi_category`, `smoking_status` and `checkup_recency` are treated as nominal even though their categories look ordered, since the underlying relationship (e.g. BMI's U-shaped risk) isn't necessarily monotonic.

In [3]:
demographic_features = ["age_group", "sex", "race_ethnicity", "education_level", "income_group",
                        "employment_status", "marital_status", "urban_rural"]
access_features = ["insurance_status", "personal_doctor", "cost_barrier", "checkup_recency"]
health_features = ["general_health", "diabetes_status", "heart_disease", "smoking_status", "bmi_category"]

M1_FEATURES = demographic_features
M2_FEATURES = M1_FEATURES + access_features
M3_FEATURES = M2_FEATURES + health_features

assert len(M1_FEATURES) == 8
assert len(M2_FEATURES) == 12
assert len(M3_FEATURES) == 17

ORDINAL_FEATURES = ["age_group", "income_group", "education_level", "general_health"]
NOMINAL_FEATURES = [f for f in M3_FEATURES if f not in ORDINAL_FEATURES]
assert len(ORDINAL_FEATURES) + len(NOMINAL_FEATURES) == 17

ORDINAL_CATEGORY_ORDER = {
    "age_group": ["45-49", "50-54", "55-59", "60-64", "65-69", "70-74",
                  "75 (source category 75-79)"],
    "income_group": ["Less than $15,000", "$15,000 to < $25,000", "$25,000 to < $35,000",
                      "$35,000 to < $50,000", "$50,000 to < $100,000", "$100,000 to < $200,000",
                      "$200,000 or more"],
    "education_level": ["Did not graduate high school", "Graduated high school",
                         "Attended college or technical school",
                         "Graduated college or technical school"],
    "general_health": ["Poor", "Fair", "Good", "Very good", "Excellent"],
}
for col, order in ORDINAL_CATEGORY_ORDER.items():
    observed = set(df[col].unique()) - {"Not reported"}
    assert observed == set(order), f"{col}: category order doesn't match observed values"

print("M1 (demographic):", M1_FEATURES)
print("M2 adds (access):", access_features)
print("M3 adds (health):", health_features)
print("Ordinal:", ORDINAL_FEATURES)
print("Nominal:", NOMINAL_FEATURES)

M1 (demographic): ['age_group', 'sex', 'race_ethnicity', 'education_level', 'income_group', 'employment_status', 'marital_status', 'urban_rural']
M2 adds (access): ['insurance_status', 'personal_doctor', 'cost_barrier', 'checkup_recency']
M3 adds (health): ['general_health', 'diabetes_status', 'heart_disease', 'smoking_status', 'bmi_category']
Ordinal: ['age_group', 'income_group', 'education_level', 'general_health']
Nominal: ['sex', 'race_ethnicity', 'employment_status', 'marital_status', 'urban_rural', 'insurance_status', 'personal_doctor', 'cost_barrier', 'checkup_recency', 'diabetes_status', 'heart_disease', 'smoking_status', 'bmi_category']


## 3. Train / calibration / test split

One split, persisted for every later notebook to reuse, stratified on the primary target. A 60/20/20 three-way split (rather than the proposal's illustrative 80/20) because calibration needs a dedicated slice the model never trains on -- with N=227,647 there's no need for cross-validated calibration to preserve sample size. `respondent_id`, `validation_group_state` (geographic validation) and the equity subgroup columns are carried through the split untouched; none of them are ever used as model features.

In [4]:
from sklearn.model_selection import train_test_split

EQUITY_COLUMNS = ["insurance_status", "income_group", "race_ethnicity", "sex", "age_group"]
AUX_COLUMNS = ["respondent_id", "validation_group_state"] + EQUITY_COLUMNS

y = df["crc_noncompliant"]
X = df[M3_FEATURES].copy()
aux = df[AUX_COLUMNS].copy()

X_train, X_temp, y_train, y_temp, aux_train, aux_temp = train_test_split(
    X, y, aux, test_size=0.4, random_state=RANDOM_SEED, stratify=y
)
X_cal, X_test, y_cal, y_test, aux_cal, aux_test = train_test_split(
    X_temp, y_temp, aux_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp
)

for name, Xs, ys in [("train", X_train, y_train), ("calibration", X_cal, y_cal), ("test", X_test, y_test)]:
    print(f"{name:12s} n={len(Xs):>7,}   non-compliance rate={ys.mean():.4f}")

assert len(X_train) + len(X_cal) + len(X_test) == len(df)

train        n=136,588   non-compliance rate=0.2648
calibration  n= 45,529   non-compliance rate=0.2648
test         n= 45,530   non-compliance rate=0.2648


**Why "calibration," not "validation":** these are two different things and it's worth being
explicit about which one this split is for. A **validation set** is for *model/hyperparameter
selection* -- comparing configurations (e.g. tree depth, regularisation strength) before
committing to one, using held-out performance to pick a winner. A **calibration set** is for
*fixing the predicted probabilities of an already-chosen, already-fit model* so that "70% predicted
risk" actually means observed non-compliance is close to 70% -- unrelated to picking between
models.

This project hasn't needed a validation split because no hyperparameter tuning has happened yet --
every model so far uses library defaults. If tuning is added later (e.g. for the gradient boosting
model), the more standard and sample-efficient approach is to do it via cross-validation *within*
the training fold (`GridSearchCV`/`RandomizedSearchCV` re-splits the training data internally),
rather than carving out a fourth fixed slice -- that keeps the calibration and test folds
completely untouched by any model-selection decision, and doesn't cost a dedicated slice of data
that a k-fold approach gets for free. The calibration fold defined below exists specifically
because the professor asked for calibration and a risk-decile table, which a validation set
wouldn't provide.

## 4. Predictor redundancy check (Cramer's V)

Proposal step (c) commits to checking predictors for correlation before modelling. All 17 predictors are categorical, so pairwise association is measured with bias-corrected Cramer's V rather than Pearson correlation. Computed on the **training fold only**, to avoid using test data for any modelling decision, even a feature-selection one.

In [5]:
from itertools import combinations
from scipy.stats import chi2_contingency

def cramers_v(a, b):
    table = pd.crosstab(a, b)
    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.sum().sum()
    r, k = table.shape
    phi2 = chi2 / n
    phi2_corr = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))
    r_corr = r - (r - 1) ** 2 / (n - 1)
    k_corr = k - (k - 1) ** 2 / (n - 1)
    denom = min(k_corr - 1, r_corr - 1)
    return np.sqrt(phi2_corr / denom) if denom > 0 else np.nan

rows = []
for f1, f2 in combinations(M3_FEATURES, 2):
    rows.append((f1, f2, cramers_v(X_train[f1], X_train[f2])))
cramers_table = (
    pd.DataFrame(rows, columns=["Feature 1", "Feature 2", "Cramers_V"])
    .sort_values("Cramers_V", ascending=False)
    .reset_index(drop=True)
)
cramers_table.to_csv(MODEL_DIR / "predictor_redundancy_cramers_v.csv", index=False)
cramers_table.head(15)

,Feature 1,Feature 2,Cramers_V
0,personal_doctor,checkup_recency,0.333184
1,age_group,employment_status,0.250034
2,insurance_status,personal_doctor,0.237593
3,education_level,income_group,0.217844
4,income_group,employment_status,0.196966
5,insurance_status,checkup_recency,0.192060
6,insurance_status,cost_barrier,0.191843
7,sex,bmi_category,0.187774
8,employment_status,general_health,0.186684
9,general_health,heart_disease,0.181837


**Reading this table:** Cramer's V ranges 0 (independent) to 1 (perfectly redundant). The strongest association is `personal_doctor` x `checkup_recency` (~0.33) -- moderate, expected (having a doctor and going for check-ups are related access behaviours), but far below the level (roughly >0.7-0.8) that would indicate the two fields are measuring almost the same thing, which was the standard already applied to justify dropping "primary coverage source" in favour of `insurance_status` at the proposal stage. No pair here clears that bar, so all 17 predictors are retained for the modelling notebook -- nothing is redundant enough to drop automatically.

## 5. Preprocessing pipeline components

Two different encodings, matched to what each model family actually needs, both wrapped in `sklearn.Pipeline` so every encoder/scaler is fit on the training fold only -- no leakage. These are defined and smoke-tested here; they are **not** attached to a trained, evaluated classifier in this notebook -- that's the modelling notebook's job.

**Logistic regression encoding:** ordinal fields become an integer rank plus a separate binary `not_reported` flag (a small custom transformer -- `OrdinalRankWithMissingFlag`), so "Not reported" doesn't sit inside the ordinal scale as if it were a value. Nominal fields are one-hot encoded (`handle_unknown="ignore"`).

**Gradient boosting encoding:** `HistGradientBoostingClassifier` supports categorical features natively -- every predictor is ordinal-encoded to integers purely as a storage format (order is irrelevant for this branch) and flagged as categorical via `categorical_features`, so the model does proper category-aware splits rather than assuming a numeric order. No one-hot needed for trees.

**Class imbalance:** both are meant to be fit with `sample_weight` from `compute_sample_weight("balanced", y_train)` rather than oversampling to 50/50 -- corrects the training loss for the 26.5% positive rate without duplicating/synthesising rows or distorting the probabilities that later get calibrated. The modelling notebook applies this; it isn't exercised here.

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier


class OrdinalRankWithMissingFlag(BaseEstimator, TransformerMixin):
    """Maps each ordinal column to its rank (0..k-1) using an explicit, pre-defined category
    order, plus a companion 0/1 'not reported' flag. 'Not reported' is never given a rank of its
    own -- it would otherwise sit at an arbitrary point on the ordinal scale.

    The category order is supplied at construction time (derived from the codebook, not fit from
    data), so `fit` only needs to record the column order it was given.
    """

    def __init__(self, category_orders):
        self.category_orders = category_orders

    def fit(self, X, y=None):
        self.columns_ = list(X.columns) if hasattr(X, "columns") else list(self.category_orders)
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns_)
        out = {}
        for col in self.columns_:
            rank_map = {cat: rank for rank, cat in enumerate(self.category_orders[col])}
            ranks = X[col].map(rank_map)
            out[f"{col}__rank"] = ranks.fillna(-1).astype(float)
            out[f"{col}__not_reported"] = ranks.isna().astype(float)
        return pd.DataFrame(out, index=X.index).values

    def get_feature_names_out(self, input_features=None):
        names = []
        for col in self.columns_:
            names += [f"{col}__rank", f"{col}__not_reported"]
        return np.array(names)


def ordinal_subset(features):
    return [f for f in features if f in ORDINAL_FEATURES]


def nominal_subset(features):
    return [f for f in features if f in NOMINAL_FEATURES]


def build_logreg_pipeline(features):
    pre = ColumnTransformer([
        ("ordinal", OrdinalRankWithMissingFlag(
            {c: ORDINAL_CATEGORY_ORDER[c] for c in ordinal_subset(features)}
        ), ordinal_subset(features)),
        ("nominal", OneHotEncoder(handle_unknown="ignore"), nominal_subset(features)),
    ])
    return Pipeline([
        ("preprocess", pre),
        ("scale", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
    ])


def build_gb_pipeline(features):
    pre = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), features),
    ])
    return Pipeline([
        ("preprocess", pre),
        ("model", HistGradientBoostingClassifier(
            categorical_features=[True] * len(features),
            random_state=RANDOM_SEED,
        )),
    ])


print("Pipeline builders defined: build_logreg_pipeline(features), build_gb_pipeline(features)")

Pipeline builders defined: build_logreg_pipeline(features), build_gb_pipeline(features)


### Smoke test

Confirms both pipelines fit and transform without error and produce the expected output shape -- a technique check, not a performance evaluation. No metric here is meant to be reported; the modelling notebook does the real fitting and evaluation.

In [7]:
for name, builder in [("logreg", build_logreg_pipeline), ("gb", build_gb_pipeline)]:
    pipe = builder(M3_FEATURES)
    pipe.fit(X_train[M3_FEATURES].iloc[:2000], y_train.iloc[:2000])
    preview = pipe.predict_proba(X_test[M3_FEATURES].iloc[:100])
    assert preview.shape == (100, 2)
    assert not np.isnan(preview).any()
    print(f"{name:8s} pipeline: fit + predict_proba OK on a 2,000-row smoke sample, "
          f"output shape {preview.shape}, no NaNs.")

print("\nBoth pipelines are ready for the modelling notebook to fit on the full training fold.")

logreg   pipeline: fit + predict_proba OK on a 2,000-row smoke sample, output shape (100, 2), no NaNs.
gb       pipeline: fit + predict_proba OK on a 2,000-row smoke sample, output shape (100, 2), no NaNs.

Both pipelines are ready for the modelling notebook to fit on the full training fold.


## 6. Geographic internal-external validation: fold structure

The professor asked about internal/external validation; a genuine Singapore holdout isn't available, so the closest available substitute is a leave-states-out `GroupKFold` on `validation_group_state`, restricted to the training fold. This section validates the **fold construction only** -- that states don't leak across folds and fold sizes are reasonably balanced. Fitting a model on each fold and reporting AUROC is the modelling notebook's job, not this one.

In [8]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
fold_rows = []
seen_states = set()
for fold, (tr_idx, va_idx) in enumerate(
    gkf.split(X_train[M3_FEATURES], y_train, groups=aux_train["validation_group_state"])
):
    held_out_states = set(aux_train["validation_group_state"].iloc[va_idx].unique())
    train_states = set(aux_train["validation_group_state"].iloc[tr_idx].unique())
    overlap = held_out_states & train_states
    assert not overlap, f"fold {fold}: states leaked across train/validation: {overlap}"
    seen_states |= held_out_states
    fold_rows.append({
        "Fold": fold, "States held out": len(held_out_states),
        "N held out": len(va_idx), "N train": len(tr_idx),
    })

fold_table = pd.DataFrame(fold_rows)
total_states = aux_train["validation_group_state"].nunique()
assert seen_states == set(aux_train["validation_group_state"].unique()), (
    "every state should be held out in exactly one fold"
)
print(f"Total distinct states/territories in training fold: {total_states}")
print("Every state is held out in exactly one fold, with no train/validation overlap within a fold.")
fold_table

Total distinct states/territories in training fold: 53
Every state is held out in exactly one fold, with no train/validation overlap within a fold.


,Fold,States held out,N held out,N train
0,0,10,27561,109027
1,1,10,27331,109257
2,2,11,27236,109352
3,3,11,27268,109320
4,4,11,27192,109396


## 7. Missingness sensitivity input: KNN imputation for `income_group` / `bmi_category`

The primary pipeline retains "Not reported" as an explicit category everywhere -- the project's committed missingness strategy. This section builds the **alternative** input the modelling notebook needs for its sensitivity comparison: `income_group` and `bmi_category` (14.9% and 5.8% not-reported respectively -- the two fields substantial enough to be worth testing) imputed via KNN instead of left as "Not reported". Whether this changes predictive performance is a modelling question; this section only builds and mechanically validates the imputed values.

Mechanics: re-encode `income_group`/`bmi_category` on their own compact ordinal scale (excluding "Not reported" from that scale entirely, so an imputed value can never round back onto the missing category by accident -- `OrdinalEncoder`'s default alphabetical category order would otherwise put "Not reported" *inside* the numeric range for `bmi_category`, which is exactly this kind of bug), fit a `KNNImputer` on a random reference sample of the training fold, and impute the rows that need it in train and test.

In [9]:
from sklearn.impute import KNNImputer

IMPUTE_CATEGORY_ORDER = {
    "income_group": ORDINAL_CATEGORY_ORDER["income_group"],
    "bmi_category": ["Underweight", "Normal weight", "Overweight", "Obese"],
}
IMPUTE_COLS = list(IMPUTE_CATEGORY_ORDER.keys())
REFERENCE_POOL_SIZE = 8_000

other_cols = [c for c in M3_FEATURES if c not in IMPUTE_COLS]
neighbour_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
Xtr_other = pd.DataFrame(neighbour_encoder.fit_transform(X_train[other_cols]),
                          columns=other_cols, index=X_train.index)
Xte_other = pd.DataFrame(neighbour_encoder.transform(X_test[other_cols]),
                          columns=other_cols, index=X_test.index)

def encode_impute_target(series, order):
    rank_map = {cat: float(i) for i, cat in enumerate(order)}
    return series.map(rank_map)  # "Not reported" isn't in the map -> becomes NaN

Xtr_target = pd.DataFrame(
    {c: encode_impute_target(X_train[c], IMPUTE_CATEGORY_ORDER[c]) for c in IMPUTE_COLS},
    index=X_train.index,
)
Xte_target = pd.DataFrame(
    {c: encode_impute_target(X_test[c], IMPUTE_CATEGORY_ORDER[c]) for c in IMPUTE_COLS},
    index=X_test.index,
)

Xtr_knn_space = pd.concat([Xtr_other, Xtr_target], axis=1)[M3_FEATURES]
Xte_knn_space = pd.concat([Xte_other, Xte_target], axis=1)[M3_FEATURES]

need_impute_train = Xtr_target.isna().any(axis=1)
need_impute_test = Xte_target.isna().any(axis=1)
print(f"Rows needing imputation -- train: {need_impute_train.sum():,}  test: {need_impute_test.sum():,}")

rng = np.random.RandomState(RANDOM_SEED)
reference_idx = rng.choice(Xtr_knn_space.index, size=REFERENCE_POOL_SIZE, replace=False)
imputer = KNNImputer(n_neighbors=5)
imputer.fit(Xtr_knn_space.loc[reference_idx])

imputed_train = pd.DataFrame(
    imputer.transform(Xtr_knn_space.loc[need_impute_train]),
    columns=M3_FEATURES, index=Xtr_knn_space.loc[need_impute_train].index,
)
imputed_test = pd.DataFrame(
    imputer.transform(Xte_knn_space.loc[need_impute_test]),
    columns=M3_FEATURES, index=Xte_knn_space.loc[need_impute_test].index,
)

knn_imputed_values = {}  # respondent_id -> {col: imputed_label}
for col in IMPUTE_COLS:
    cats = np.array(IMPUTE_CATEGORY_ORDER[col])
    tr_missing = X_train.index[X_train[col] == "Not reported"]
    te_missing = X_test.index[X_test[col] == "Not reported"]
    rounded_tr = imputed_train.loc[tr_missing, col].round().clip(0, len(cats) - 1).astype(int)
    rounded_te = imputed_test.loc[te_missing, col].round().clip(0, len(cats) - 1).astype(int)
    knn_imputed_values[col] = pd.concat([
        pd.Series(cats[rounded_tr.values], index=tr_missing),
        pd.Series(cats[rounded_te.values], index=te_missing),
    ])

print("KNN imputation complete for both columns across train + test.")

Rows needing imputation -- train: 24,747  test: 8,518


KNN imputation complete for both columns across train + test.


**Mechanical validation** -- confirms every previously-"Not reported" cell now has a real category, and that the imputed distribution is a plausible population, not an artifact (e.g. everything collapsing onto one category, which is exactly what the alphabetical-encoding bug would have caused for `bmi_category`).

In [10]:
for col in IMPUTE_COLS:
    imputed_series = knn_imputed_values[col]
    assert imputed_series.isin(IMPUTE_CATEGORY_ORDER[col]).all(), (
        f"{col}: imputed values include a category outside the valid set"
    )
    assert not imputed_series.eq("Not reported").any()
    print(f"{col}: {len(imputed_series):,} cells imputed, 0 remaining 'Not reported'.")
    print(imputed_series.value_counts(normalize=True).round(3).rename("share of imputed rows"))
    print()

# Save a lookup table (respondent_id -> imputed value) for the modelling notebook to merge in.
lookup_rows = []
for col in IMPUTE_COLS:
    s = knn_imputed_values[col]
    resp_ids = pd.concat([aux_train["respondent_id"], aux_test["respondent_id"]]).loc[s.index]
    lookup_rows.append(pd.DataFrame({
        "respondent_id": resp_ids.values, "column": col, "imputed_value": s.values,
    }))
knn_lookup = pd.concat(lookup_rows, ignore_index=True)
knn_lookup.to_csv(MODEL_DIR / "knn_imputed_income_bmi_lookup.csv", index=False)
print(f"Saved {len(knn_lookup):,} imputed values to knn_imputed_income_bmi_lookup.csv "
      "for the modelling notebook's own sensitivity comparison.")

income_group: 27,202 cells imputed, 0 remaining 'Not reported'.
$50,000 to < $100,000     0.361
$35,000 to < $50,000      0.233
$100,000 to < $200,000    0.223
$25,000 to < $35,000      0.122
$15,000 to < $25,000      0.044
$200,000 or more          0.013
Less than $15,000         0.003
Name: share of imputed rows, dtype: float64

bmi_category: 10,675 cells imputed, 0 remaining 'Not reported'.
Overweight       0.729
Obese            0.163
Normal weight    0.108
Name: share of imputed rows, dtype: float64



Saved 37,877 imputed values to knn_imputed_income_bmi_lookup.csv for the modelling notebook's own sensitivity comparison.


## 8. Persist processed artifacts

Everything the modelling notebook needs to start from, without re-deriving it: the row-level split assignment (by `respondent_id`), the Cramer's V table, and the KNN-imputed lookup table saved above. The preprocessing pipeline code (`OrdinalRankWithMissingFlag`, `build_logreg_pipeline`, `build_gb_pipeline`, the domain feature lists and category orders) is redefined identically in the modelling notebook, the same way notebook 2 redefines notebook 1's domain lists -- each notebook is self-contained and runnable on its own, consistent with this project's existing convention.

In [11]:
split_assignment = pd.concat([
    aux_train[["respondent_id"]].assign(split="train"),
    aux_cal[["respondent_id"]].assign(split="calibration"),
    aux_test[["respondent_id"]].assign(split="test"),
]).reset_index(drop=True)
split_assignment.to_csv(MODEL_DIR / "train_calibration_test_split.csv", index=False)

assert split_assignment["respondent_id"].is_unique
assert len(split_assignment) == len(df)

print("Artifacts written to:", MODEL_DIR)
for p in sorted(MODEL_DIR.glob("*")):
    print(" -", p.name)

Artifacts written to: ../data/processed/modelling
 - ablation_results.csv
 - baseline_logreg_odds_ratios.csv
 - baseline_logreg_results.csv
 - ffnn_permutation_importance.csv
 - ffnn_results.csv
 - gb_permutation_importance.csv
 - gb_results.csv
 - geographic_validation.csv
 - knn_imputed_income_bmi_lookup.csv
 - model_comparison_table.csv
 - model_registry_manifest.csv
 - modelling_run_summary.json
 - models
 - never_screened_permutation_importance.csv
 - overdue_given_screened_permutation_importance.csv
 - predictor_redundancy_cramers_v.csv
 - rf_permutation_importance.csv
 - rf_results.csv
 - rf_test_probas.npz
 - rf_tuned_best_params.json
 - risk_decile_table.csv
 - sensitivity_knn_vs_not_reported.csv
 - subgroup_equity_performance.csv
 - train_calibration_test_split.csv
 - twostage_model_registry_manifest.csv
 - twostage_models


## 9. Handoff to the modelling notebook

**Ready to use, no re-derivation needed:**
- `data/processed/modelling/train_calibration_test_split.csv` -- respondent-level 60/20/20 split assignment.
- `data/processed/modelling/predictor_redundancy_cramers_v.csv` -- full pairwise Cramer's V table; nothing flagged for removal.
- `data/processed/modelling/knn_imputed_income_bmi_lookup.csv` -- KNN-imputed `income_group`/`bmi_category` values, ready to merge onto the primary dataset for the sensitivity branch.

**Validated, ready to build on:**
- The M1 (8) / M2 (12) / M3 (17) predictor domains and the ordinal/nominal encoding split.
- The logistic-regression and gradient-boosting preprocessing pipelines -- fit/transform smoke-tested, no errors, correct output shape.
- The geographic `GroupKFold` structure -- 5 folds, no state leakage across train/validation within a fold, every state held out exactly once.

**Explicitly not done here, and next:** fit logistic regression (the interpretable baseline) and gradient boosting on M1/M2/M3, using `sample_weight` for class imbalance rather than oversampling; evaluate on the held-out test fold; calibrate the leading model and build the risk-decile table; run the geographic-fold evaluation for real; report subgroup equity performance; and compare the primary vs. KNN-imputed sensitivity branch on actual predictive performance. Baseline logistic regression is next.